In [43]:
# downloading spaCy and ua model
!pip install -U pip setuptools wheel
!pip install -U spacy
!python -m spacy download uk_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 25.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [51]:
# imports and gold set
import pandas as pd
import json, spacy, os, re
from collections import Counter, defaultdict
from spacy.pipeline import EntityRuler
from spacy.tokens import Span
from spacy.language import Language

out_dir = "/content/eval_output"
os.makedirs(out_dir, exist_ok=True)

gold_path = "/content/evaluation_set.json"
with open(gold_path, "r", encoding="utf-8") as f:
    gold = json.load(f)

print("Loaded", len(gold), "records from", gold_path)

Loaded 20 records from /content/evaluation_set.json


In [45]:
# helper functions
def span_key(s):
    return (s["start"], s["end"], s["label"].upper())

def prf(tp, fp, fn):
    prec = tp/(tp+fp) if tp+fp>0 else 0.0
    rec = tp/(tp+fn) if tp+fn>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
    return prec, rec, f1

def evaluate_predictions(gold_records, preds_records):
    metrics = {"TP":0, "FP":0, "FN":0}
    per_label = defaultdict(lambda: {"TP":0, "FP":0, "FN":0})
    details = []
    for grec, prec in zip(gold_records, preds_records):
        gold_spans = [{"start":e["start"], "end":e["end"], "label":e["label"].upper(), "text":e["text"]} for e in grec["entities"]]
        pred_spans = [{"start":p["start"], "end":p["end"], "label":p["label"].upper(), "text":p["text"]} for p in prec]
        gkeys = set(span_key(s) for s in gold_spans)
        pkeys = set(span_key(s) for s in pred_spans)
        tp = gkeys & pkeys
        fp = pkeys - gkeys
        fn = gkeys - pkeys
        metrics["TP"] += len(tp)
        metrics["FP"] += len(fp)
        metrics["FN"] += len(fn)
        for k in tp: per_label[k[2]]["TP"] += 1
        for k in fp: per_label[k[2]]["FP"] += 1
        for k in fn: per_label[k[2]]["FN"] += 1
        details.append({"text_id": grec["text_id"], "text": grec["text"], "gold": gold_spans, "preds": pred_spans, "tp": len(tp), "fp": len(fp), "fn": len(fn)})
    return metrics, per_label, details

In [46]:
# baseline inference
nlp = spacy.load("uk_core_news_sm")
print("Loaded model:", nlp.meta.get("name", "uk_core_news_sm"))

baseline_preds = []
for rec in gold:
    doc = nlp(rec["text"])
    preds = []
    for e in doc.ents:
        preds.append({"start": e.start_char, "end": e.end_char, "label": e.label_.upper(), "text": e.text})
    baseline_preds.append(preds)

metrics_base, per_label_base, details_base = evaluate_predictions(gold, baseline_preds)
prec_base, rec_base, f1_base = prf(metrics_base["TP"], metrics_base["FP"], metrics_base["FN"])

summary_base = {"overall": {"TP":metrics_base["TP"], "FP":metrics_base["FP"], "FN":metrics_base["FN"], "precision":prec_base, "recall":rec_base, "f1":f1_base}, "per_label": {}}
for lab, vals in per_label_base.items():
    p,r,f = prf(vals["TP"], vals["FP"], vals["FN"])
    summary_base["per_label"][lab] = {"TP":vals["TP"], "FP":vals["FP"], "FN":vals["FN"], "precision":p, "recall":r, "f1":f}

# save baseline outputs
with open(os.path.join(out_dir, "baseline_predictions.json"), "w", encoding="utf-8") as f:
    json.dump(details_base, f, ensure_ascii=False, indent=2)
with open(os.path.join(out_dir, "baseline_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary_base, f, ensure_ascii=False, indent=2)

print("Baseline overall: TP={}, FP={}, FN={}, precision={:.3f}, recall={:.3f}, f1={:.3f}".format(
    metrics_base["TP"], metrics_base["FP"], metrics_base["FN"], prec_base, rec_base, f1_base))

# print examples
print("\nBaseline examples:")
for d in details_base[:20]:
    print("---")
    print("Text ID:", d["text_id"])
    print(d["text"])
    print("Predicted:", d["preds"])
    print("Expected:", d["gold"])

Loaded model: core_news_sm
Baseline overall: TP=33, FP=7, FN=33, precision=0.825, recall=0.500, f1=0.623

Baseline examples:
---
Text ID: 1
у центрі миколаївської області збито ракету у центрі миколаївської області збито ракету
Predicted: [{'start': 9, 'end': 30, 'label': 'LOC', 'text': 'миколаївської області'}, {'start': 53, 'end': 74, 'label': 'LOC', 'text': 'миколаївської області'}]
Expected: [{'start': 9, 'end': 30, 'label': 'LOC', 'text': 'миколаївської області'}, {'start': 53, 'end': 74, 'label': 'LOC', 'text': 'миколаївської області'}]
---
Text ID: 2
україна скоротила "лнр" від водопостачання україна відрізала "лнр" від водопостачання.
Predicted: [{'start': 0, 'end': 7, 'label': 'LOC', 'text': 'україна'}, {'start': 19, 'end': 22, 'label': 'ORG', 'text': 'лнр'}, {'start': 43, 'end': 50, 'label': 'LOC', 'text': 'україна'}, {'start': 62, 'end': 65, 'label': 'ORG', 'text': 'лнр'}]
Expected: [{'start': 0, 'end': 7, 'label': 'LOC', 'text': 'україна'}, {'start': 19, 'end': 22, 'label':

In [47]:
# EntityRuler для простих однотокенних патернів (PRODUCT, TITLE)

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")

ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = []

# PRODUCT однотокенні варіанти
prod_list = ["су-34", "су34", "рсзв", "mh17", "mh-17", "mh 17"]
for p in prod_list:
    patterns.append({"label":"PRODUCT", "pattern":[{"LOWER": p.lower()}]})
# короткі фрази
patterns.append({"label":"PRODUCT", "pattern":[{"LOWER":"рятувальне"},{"LOWER":"судно"},{"LOWER":"\"сапфір\""}]})

# TITLE словник (один- та багатотокенні)
titles = ["генпрокурор","генсек","президент","експрезидент","очільник","міністр",
          "постійний представник","голова","командувач","директор","постійний","представник"]
for t in titles:
    toks = t.split()
    patterns.append({"label":"TITLE", "pattern":[{"LOWER":tok} for tok in toks]})

ruler.add_patterns(patterns)
print("EntityRuler patterns added:", len(patterns))

EntityRuler patterns added: 19


In [48]:
# реєстрація regex_ner як spaCy component (char-level matching)

# Патерни на рівні рядка
money_patterns = [
    r"\b\d{1,3}(?:[ \,]\d{3})*(?:-\d{1,3}(?:[ \,]\d{3})*)?\s*(?:грн\.?|гривень|гривні|uah)\b",
    r"\b\d+(?:-\d+)?\s*(?:грн\.?|гривень|гривні|uah)\b",
    r"\b\d{1,3}(?:[ \,]\d{3})*(?:-\d{1,3}(?:[ \,]\d{3})*)?\s*(?:\$|usd|доларів|долар|долари)\b",
    r"\b\d+(?:-\d+)?\s*(?:€|eur|євро)\b"
]
percent_patterns = [r"\d{1,3}(?:-\d{1,3})?%", r"\d{1,3}(?:-\d{1,3})?\s*%", r"\b\d+(?:-\d+)?\s*%\b", r"\b\d+(?:-\d+)?%\b"]
quantity_patterns = [
    r"\b\d{1,3}(?:[ \,]\d{3})*\+?\s*(?:км|км\.|кілометрів|кілометр|кілометри)\b",
    r"\b\d{1,3}(?:[ \,]\d{3})*\s*(?:м|м\.|метрів|метр)\b",
    r"\d{1,3}(?:[ \u00A0]\d{3})+",  # ловить 100 000 з пробілом або не‑переносним пробілом
    r"\d{1,3}(?:[ \,]\d{3})*\s*(?:км|кілометрів|метрів|м)\b"
]
months = ["січня","лютого","березня","квітня","травня","червня","липня","серпня","вересня","жовтня","листопада","грудня"]
date_patterns = [r"\b\d{1,2}\s+(?:%s)\b" % ("|".join(months)), r"\b(19|20)\d{2}\b", r"\b'\d{2}\b"]
days = ["понеділок","вівторок","середа","четвер","п'ятниця","пятниця","субота","неділя"]
day_pattern = r"\b(?:%s)\b" % ("|".join(days))
multi_prod_patterns = [r"рятувальне\s+судно\s*(?:\"|«)?\s*сапфір(?:\"|»)?", r"\bрятувальне\s+судно\s+сапфір\b"]

regex_rules = []
for rx in money_patterns:
    regex_rules.append(("MONEY", re.compile(rx, flags=re.IGNORECASE)))
for rx in percent_patterns:
    regex_rules.append(("PERCENT", re.compile(rx, flags=re.IGNORECASE)))
for rx in quantity_patterns:
    regex_rules.append(("QUANTITY", re.compile(rx, flags=re.IGNORECASE)))
for rx in date_patterns:
    regex_rules.append(("DATE", re.compile(rx, flags=re.IGNORECASE)))
regex_rules.append(("DATE", re.compile(day_pattern, flags=re.IGNORECASE)))
for rx in multi_prod_patterns:
    regex_rules.append(("PRODUCT", re.compile(rx, flags=re.IGNORECASE)))

# Реєстрація regex компонента через декоратор
@Language.component("regex_ner")
def regex_ner_component(doc):
    new_ents = list(doc.ents)
    text = doc.text
    for label, pattern in regex_rules:
        for m in pattern.finditer(text):
            start_char, end_char = m.start(), m.end()
            # уникаємо точних дублікатів
            duplicate = False
            for ent in new_ents:
                if ent.start_char == start_char and ent.end_char == end_char and ent.label_ == label:
                    duplicate = True
                    break
            if duplicate:
                continue
            # намагаємось створити span через char_span з alignment_mode="expand"
            span = doc.char_span(start_char, end_char, label=label, alignment_mode="expand")
            if span is None:
                # fallback: знайти токени, що перекриваються
                token_start = None
                token_end = None
                for i, token in enumerate(doc):
                    if token.idx <= start_char < token.idx + len(token):
                        token_start = i
                    if token.idx < end_char <= token.idx + len(token):
                        token_end = i + 1
                if token_start is None:
                    token_start = 0
                if token_end is None:
                    token_end = len(doc)
                span = Span(doc, token_start, token_end, label=label)
            new_ents.append(span)
    doc.ents = tuple(new_ents)
    return doc

# Додаємо компонент у pipeline після entity_ruler
if "regex_ner" in nlp.pipe_names:
    nlp.remove_pipe("regex_ner")
nlp.add_pipe("regex_ner", after="entity_ruler")
print("Added regex_ner component; pipeline:", nlp.pipe_names)

Added regex_ner component; pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'regex_ner', 'ner']


In [49]:
# hybrid inference, evaluation
import json
from collections import defaultdict

def span_key(s): return (s["start"], s["end"], s["label"].upper())

def evaluate_predictions(gold_records, preds_records):
    metrics = {"TP":0, "FP":0, "FN":0}
    per_label = defaultdict(lambda: {"TP":0, "FP":0, "FN":0})
    details = []
    for grec, prec in zip(gold_records, preds_records):
        gold_spans = [{"start":e["start"], "end":e["end"], "label":e["label"].upper(), "text":e["text"]} for e in grec["entities"]]
        pred_spans = [{"start":p["start"], "end":p["end"], "label":p["label"].upper(), "text":p["text"]} for p in prec]
        gkeys = set(span_key(s) for s in gold_spans)
        pkeys = set(span_key(s) for s in pred_spans)
        tp = gkeys & pkeys
        fp = pkeys - gkeys
        fn = gkeys - pkeys
        metrics["TP"] += len(tp); metrics["FP"] += len(fp); metrics["FN"] += len(fn)
        for k in tp: per_label[k[2]]["TP"] += 1
        for k in fp: per_label[k[2]]["FP"] += 1
        for k in fn: per_label[k[2]]["FN"] += 1
        details.append({"text_id": grec["text_id"], "text": grec["text"], "gold": gold_spans, "preds": pred_spans, "tp": len(tp), "fp": len(fp), "fn": len(fn)})
    return metrics, per_label, details

def prf(tp,fp,fn):
    prec = tp/(tp+fp) if tp+fp>0 else 0.0
    rec = tp/(tp+fn) if tp+fn>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0.0
    return prec, rec, f1

# Run hybrid inference
hybrid_preds = []
for rec in gold:
    doc = nlp(rec["text"])
    preds = []
    for e in doc.ents:
        preds.append({"start": e.start_char, "end": e.end_char, "label": e.label_.upper(), "text": e.text})
    hybrid_preds.append(preds)

metrics_h, per_label_h, details_h = evaluate_predictions(gold, hybrid_preds)
prec_h, rec_h, f1_h = prf(metrics_h["TP"], metrics_h["FP"], metrics_h["FN"])

summary_h = {"overall": {"TP":metrics_h["TP"], "FP":metrics_h["FP"], "FN":metrics_h["FN"], "precision":prec_h, "recall":rec_h, "f1":f1_h}, "per_label": {}}
for lab, vals in per_label_h.items():
    p,r,f = prf(vals["TP"], vals["FP"], vals["FN"])
    summary_h["per_label"][lab] = {"TP":vals["TP"], "FP":vals["FP"], "FN":vals["FN"], "precision":p, "recall":r, "f1":f}

# Save
out_dir = "/content/eval_output"
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "hybrid_predictions.json"), "w", encoding="utf-8") as f:
    json.dump(details_h, f, ensure_ascii=False, indent=2)
with open(os.path.join(out_dir, "hybrid_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary_h, f, ensure_ascii=False, indent=2)

print("Hybrid overall: TP={}, FP={}, FN={}, precision={:.3f}, recall={:.3f}, f1={:.3f}".format(
    metrics_h["TP"], metrics_h["FP"], metrics_h["FN"], prec_h, rec_h, f1_h))

# Print examples
print("\nHybrid examples:")
for d in details_h[:20]:
    print("---")
    print("Text ID:", d["text_id"])
    print(d["text"])
    print("Predicted:", d["preds"])
    print("Expected:", d["gold"])

Hybrid overall: TP=52, FP=5, FN=14, precision=0.912, recall=0.788, f1=0.846

Hybrid examples:
---
Text ID: 1
у центрі миколаївської області збито ракету у центрі миколаївської області збито ракету
Predicted: [{'start': 9, 'end': 30, 'label': 'LOC', 'text': 'миколаївської області'}, {'start': 53, 'end': 74, 'label': 'LOC', 'text': 'миколаївської області'}]
Expected: [{'start': 9, 'end': 30, 'label': 'LOC', 'text': 'миколаївської області'}, {'start': 53, 'end': 74, 'label': 'LOC', 'text': 'миколаївської області'}]
---
Text ID: 2
україна скоротила "лнр" від водопостачання україна відрізала "лнр" від водопостачання.
Predicted: [{'start': 0, 'end': 7, 'label': 'LOC', 'text': 'україна'}, {'start': 19, 'end': 22, 'label': 'ORG', 'text': 'лнр'}, {'start': 43, 'end': 50, 'label': 'LOC', 'text': 'україна'}, {'start': 62, 'end': 65, 'label': 'ORG', 'text': 'лнр'}]
Expected: [{'start': 0, 'end': 7, 'label': 'LOC', 'text': 'україна'}, {'start': 19, 'end': 22, 'label': 'ORG', 'text': 'лнр'}, {'start

In [50]:
# порівняння та error analysis
# Завантажити baseline_metrics.json якщо потрібно
with open(os.path.join(out_dir, "baseline_metrics.json"), "r", encoding="utf-8") as f:
    summary_base = json.load(f)
with open(os.path.join(out_dir, "hybrid_metrics.json"), "r", encoding="utf-8") as f:
    summary_h = json.load(f)

print("Baseline overall:", summary_base["overall"])
print("Hybrid overall:", summary_h["overall"])

# per-label comparison
labels = set(list(summary_base.get("per_label", {}).keys()) + list(summary_h.get("per_label", {}).keys()))
print("\nPer-label comparison (baseline -> hybrid):")
for lab in sorted(labels):
    b = summary_base.get("per_label", {}).get(lab, {"TP":0,"FP":0,"FN":0})
    h = summary_h.get("per_label", {}).get(lab, {"TP":0,"FP":0,"FN":0})
    print(f"{lab}: baseline TP={b['TP']} FN={b['FN']} FP={b['FP']}  -> hybrid TP={h['TP']} FN={h['FN']} FP={h['FP']}")

# Error analysis: зібрати до 50 помилок з hybrid details
errors = []
for det in details_h:
    gold_spans = det["gold"]
    pred_spans = det["preds"]
    gkeys = set(span_key(s) for s in gold_spans)
    pkeys = set(span_key(s) for s in pred_spans)
    fn = gkeys - pkeys
    fp = pkeys - gkeys
    for k in fn:
        start,end,label = k
        context = det["text"][max(0,start-30):min(len(det["text"]), end+30)]
        expected = next((s for s in gold_spans if s["start"]==start and s["end"]==end and s["label"]==label), None)
        errors.append({"text_id": det["text_id"], "context": context, "expected": expected, "predicted": None, "error_type": "FN", "category": "", "explanation": ""})
    for k in fp:
        start,end,label = k
        context = det["text"][max(0,start-30):min(len(det["text"]), end+30)]
        predicted = next((s for s in pred_spans if s["start"]==start and s["end"]==end and s["label"]==label), None)
        errors.append({"text_id": det["text_id"], "context": context, "expected": None, "predicted": predicted, "error_type": "FP", "category": "", "explanation": ""})

errors = errors[:50]
with open(os.path.join(out_dir, "error_analysis.json"), "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print("\nSaved error_analysis.json with", len(errors), "items")
# Print first 15 errors
print("\nFirst 15 errors:")
for e in errors[:15]:
    print("---")
    print("Text ID:", e["text_id"])
    print("Context:", e["context"])
    print("Expected:", e["expected"])
    print("Predicted:", e["predicted"])
    print("Type:", e["error_type"])

Baseline overall: {'TP': 33, 'FP': 7, 'FN': 33, 'precision': 0.825, 'recall': 0.5, 'f1': 0.6226415094339622}
Hybrid overall: {'TP': 52, 'FP': 5, 'FN': 14, 'precision': 0.9122807017543859, 'recall': 0.7878787878787878, 'f1': 0.8455284552845528}

Per-label comparison (baseline -> hybrid):
DATE: baseline TP=0 FN=2 FP=0  -> hybrid TP=2 FN=0 FP=0
LOC: baseline TP=21 FN=2 FP=2  -> hybrid TP=21 FN=2 FP=2
MONEY: baseline TP=0 FN=1 FP=0  -> hybrid TP=1 FN=0 FP=0
ORG: baseline TP=5 FN=8 FP=4  -> hybrid TP=5 FN=8 FP=2
PER: baseline TP=7 FN=4 FP=1  -> hybrid TP=7 FN=4 FP=1
PERCENT: baseline TP=0 FN=2 FP=0  -> hybrid TP=2 FN=0 FP=0
PRODUCT: baseline TP=0 FN=8 FP=0  -> hybrid TP=8 FN=0 FP=0
QUANTITY: baseline TP=0 FN=4 FP=0  -> hybrid TP=4 FN=0 FP=0
TITLE: baseline TP=0 FN=2 FP=0  -> hybrid TP=2 FN=0 FP=0

Saved error_analysis.json with 19 items

First 15 errors:
---
Text ID: 3
Context: повітряна тривога в низці областей! повітряна тривога в низці об
Expected: {'start': 26, 'end': 34, 'label': 'LOC'

In [52]:
# audit_summary_lab10.md на основі файлів metrics та error_analysis


out_dir = "/content/eval_output"
baseline_path = os.path.join(out_dir, "baseline_metrics.json")
hybrid_path = os.path.join(out_dir, "hybrid_metrics.json")
errors_path = os.path.join(out_dir, "error_analysis.json")
out_md = "/content/audit_summary_lab10.md"

# Завантажити файли
with open(baseline_path, "r", encoding="utf-8") as f:
    baseline = json.load(f)
with open(hybrid_path, "r", encoding="utf-8") as f:
    hybrid = json.load(f)
with open(errors_path, "r", encoding="utf-8") as f:
    errors = json.load(f)

# Підрахунок помилок по категоріям
cat_counter = Counter()
type_counter = Counter()
for e in errors:
    cat = e.get("category") or "Uncategorized"
    etype = e.get("error_type") or "UNK"
    cat_counter[cat] += 1
    type_counter[etype] += 1

top_cats = cat_counter.most_common(6)

# Підготовка тексту аудиту
md_lines = []
md_lines.append("# Audit Summary Lab 10\n")
md_lines.append("## Короткий summary\n")
md_lines.append(f"- **Pipeline використано:** spaCy `uk_core_news_sm` + EntityRuler + regex‑based component (char‑level).\n")
md_lines.append("- **Важливі сутності в задачі:** **LOC**, **ORG**, **PER**, **PRODUCT**, **QUANTITY/DISTANCE**, **MONEY**, **PERCENT**, **DATE**, **TITLE**.\n")

md_lines.append("## Результати (ключові метрики)\n")
md_lines.append("### Baseline\n")
b = baseline["overall"]
md_lines.append(f"- **TP:** {b['TP']}, **FP:** {b['FP']}, **FN:** {b['FN']}\n")
md_lines.append(f"- **Precision:** {b['precision']:.3f}, **Recall:** {b['recall']:.3f}, **F1:** {b['f1']:.3f}\n")

md_lines.append("### Hybrid (pipeline + rules)\n")
h = hybrid["overall"]
md_lines.append(f"- **TP:** {h['TP']}, **FP:** {h['FP']}, **FN:** {h['FN']}\n")
md_lines.append(f"- **Precision:** {h['precision']:.3f}, **Recall:** {h['recall']:.3f}, **F1:** {h['f1']:.3f}\n")

md_lines.append("## Що baseline знаходив добре\n")
md_lines.append("- **LOC**: географічні назви в більшості випадків (висока точність і recall для LOC).\n")
md_lines.append("- **PER**: деякі імена/прізвища модель розпізнавала коректно.\n")

md_lines.append("## Які доменні / регулярні сутності baseline пропускав\n")
md_lines.append("- **PRODUCT, QUANTITY, PERCENT, DATE, MONEY, TITLE** — практично не знаходились baseline'ом; потребували правил/regex.\n")

md_lines.append("## Які rules були додані\n")
md_lines.append("- **EntityRuler**: словникові патерни для PRODUCT (су-34, mh17, рятувальне судно, сапфір) та TITLE (генпрокурор, міністр, президент тощо).\n")
md_lines.append("- **Regex component (char‑level)**: правила для **MONEY** (грн, $, €; діапазони, тисячні роздільники), **PERCENT** (50-60%), **QUANTITY** (100+ км, 100 000), **DATE** (день+місяць, роки), multiword PRODUCT (рятувальне судно \"сапфір\").\n")

md_lines.append("## Що вони реально покращили\n")
md_lines.append(f"- **TP** збільшився з {b['TP']} до {h['TP']} (+{h['TP']-b['TP']}).\n")
md_lines.append(f"- **FN** зменшився з {b['FN']} до {h['FN']} (−{b['FN']-h['FN']}).\n")
md_lines.append(f"- **Precision** змінилась з {b['precision']:.3f} до {h['precision']:.3f}.\n")
md_lines.append("- Правила повністю покрили доменні категорії: PRODUCT, QUANTITY, PERCENT, DATE, MONEY, TITLE (TP для цих класів у hybrid = 1.0 recall).\n")

md_lines.append("## Які категорії помилок були наймасовішими\n")
md_lines.append("- **Organization acronym missing / Abbreviation not recognized** (НАТО, ЄС, ДНР, ППО) — багато FN та деякі label mismatches.\n")
md_lines.append("- **Person surname missing** (Пелосі, Кадирова, Лаврова) — FN для персональних імен.\n")
md_lines.append("- **Complex organization name / Partial span** — модель іноді розбиває складні ORG на частини (FP часткові спани).\n")
md_lines.append("\nTop error categories (count):\n")
for cat, cnt in top_cats:
    md_lines.append(f"- {cat}: {cnt}\n")

md_lines.append("## Що б робили далі\n")
md_lines.append("- **Розширити словники** для ORG та PER (акроніми та часті прізвища) — закрити залишкові FN.\n")
md_lines.append("- **Додати post‑processing фільтри**: видаляти короткі вкладені енти, виправляти злиття span (наприклад 'лаврова єс').\n")
md_lines.append("- **Звузити деякі regex** (за потреби) щоб зменшити FP по ORG/LOC label mismatches.\n")
md_lines.append("- **Зібрати більше прикладів** для ORG/PER у різних відмінках і додати до тренувального набору або до правил.\n")

# Записати файл
with open(out_md, "w", encoding="utf-8") as f:
    f.writelines(line + ("\n" if not line.endswith("\n") else "") for line in md_lines)

print("Saved", out_md)


Saved /content/audit_summary_lab10.md
